# ONS Postcode Directory — Extract Postcode, Lat/Lng, Place Names

This notebook explores the ONS Postcode Directory files and extracts clean data for the PocketBase `postcodes` and `places` collections.

**Data sources:**
- `ONSPD_AUG_2025/Data/ONSPD_AUG_2025_UK.csv` — 2.7M rows, 1.4 GB (main postcode data)
- `Online_ONS_Postcode_Directory_Live_1351916940913356513.xlsx` — 452 MB (alternative format)
- `ONSPD_AUG_2025/Documents/` — Lookup tables for ward, district, parish, built-up area names

**Strategy:** Use chunked reads (pandas `chunksize`) because these files are too large to fit in memory at once.

In [1]:
# Install dependencies (run once)
%pip install pandas openpyxl

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path

BASE = Path(r"c:\Users\roman\Documents\projects\UK_Music_Experiences")
CSV_PATH = BASE / "ONSPD_AUG_2025" / "Data" / "ONSPD_AUG_2025_UK.csv"
XLSX_PATH = BASE / "Online_ONS_Postcode_Directory_Live_1351916940913356513.xlsx"
DOCS = BASE / "ONSPD_AUG_2025" / "Documents"

print(f"CSV exists: {CSV_PATH.exists()}  ({CSV_PATH.stat().st_size / 1e9:.2f} GB)")
print(f"XLSX exists: {XLSX_PATH.exists()}  ({XLSX_PATH.stat().st_size / 1e9:.2f} GB)")

CSV exists: True  (1.45 GB)
XLSX exists: True  (0.47 GB)


## 1. Explore the CSV (ONSPD_AUG_2025_UK.csv)

Read just the first 5 rows to understand the column structure. The CSV has 52 columns — most are administrative area codes. We only need a few.

In [3]:
# Peek at first 5 rows — only reads a tiny slice of the 1.4 GB file
sample = pd.read_csv(CSV_PATH, nrows=5)
print(f"Columns ({len(sample.columns)}):")
print(list(sample.columns))
sample

Columns (53):
['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'cty25cd', 'ced25cd', 'lad25cd', 'wd25cd', 'parncp25cd', 'usrtypind', 'east1m', 'north1m', 'gridind', 'hlth19cd', 'nhser24cd', 'ctry25cd', 'rgn25cd', 'ssr95cd', 'pcon24cd', 'eer20cd', 'educ23cd', 'ttwa15cd', 'pco19cd', 'itl25cd', 'wdstl05cd', 'oa01cd', 'wdcas03cd', 'npark16cd', 'lsoa01cd', 'msoa01cd', 'ruc01ind', 'oac01ind', 'oa11cd', 'lsoa11cd', 'msoa11cd', 'wz11cd', 'sicbl24cd', 'bua24cd', 'ruc11ind', 'oac11ind', 'lat', 'long', 'lep21cd1', 'lep21cd2', 'pfa23cd', 'imd20ind', 'cal24cd', 'icb23cd', 'oa21cd', 'lsoa21cd', 'msoa21cd', 'ruc21ind']


,pcd7,pcd8,pcds,dointr,doterm,cty25cd,ced25cd,lad25cd,wd25cd,parncp25cd,...,lep21cd1,lep21cd2,pfa23cd,imd20ind,cal24cd,icb23cd,oa21cd,lsoa21cd,msoa21cd,ruc21ind
0,AB1 0AA,AB1 0AA,AB1 0AA,198001,199606,S99999999,S99999999,S12000033,S13002843,S99999999,...,S99999999,NaN,S23000009,6715,S99999999,S99999999,S00137176,S01013490,S02002516,1
1,AB1 0AB,AB1 0AB,AB1 0AB,198001,199606,S99999999,S99999999,S12000033,S13002843,S99999999,...,S99999999,NaN,S23000009,6715,S99999999,S99999999,S00137176,S01013490,S02002516,1
2,AB1 0AD,AB1 0AD,AB1 0AD,198001,199606,S99999999,S99999999,S12000033,S13002843,S99999999,...,S99999999,NaN,S23000009,6715,S99999999,S99999999,S00137176,S01013490,S02002516,1
3,AB1 0AE,AB1 0AE,AB1 0AE,199402,199606,S99999999,S99999999,S12000034,S13002864,S99999999,...,S99999999,NaN,S23000009,5069,S99999999,S99999999,S00138891,S01013856,S02002577,5
4,AB1 0AF,AB1 0AF,AB1 0AF,199012,199207,S99999999,S99999999,S12000033,S13002843,S99999999,...,S99999999,NaN,S23000009,6253,S99999999,S99999999,S00137241,S01013487,S02002515,1


In [4]:
# The columns we care about:
#   pcds     — postcode in readable format (e.g. "AB1 0AA")
#   lat      — latitude (WGS84)
#   long     — longitude (WGS84)
#   doterm   — date of termination. Empty = live postcode. Non-empty = terminated.
#   lad25cd  — local authority district code (joins to LAD lookup → district name)
#   wd25cd   — ward code (joins to WD lookup → ward/neighbourhood name)
#   bua24cd  — built-up area code (joins to BUA lookup → town/city name)
#   parncp25cd — parish code (joins to PARNCP lookup → parish name)

cols_of_interest = ["pcds", "lat", "long", "doterm", "lad25cd", "wd25cd", "bua24cd", "parncp25cd"]
sample[cols_of_interest]

,pcds,lat,long,doterm,lad25cd,wd25cd,bua24cd,parncp25cd
0,AB1 0AA,57.101459,-2.242858,199606,S12000033,S13002843,S99999999,S99999999
1,AB1 0AB,57.102539,-2.246315,199606,S12000033,S13002843,S99999999,S99999999
2,AB1 0AD,57.100541,-2.248349,199606,S12000033,S13002843,S99999999,S99999999
3,AB1 0AE,57.084429,-2.255714,199606,S12000034,S13002864,S99999999,S99999999
4,AB1 0AF,57.096641,-2.258109,199207,S12000033,S13002843,S99999999,S99999999


## 2. Explore the XLSX (Live Directory)

Read only the first 5 rows to see if it has different/friendlier columns. This file is 452 MB so we use `nrows` to avoid loading the whole thing.

In [5]:
# WARNING: Even reading 5 rows from a 452 MB xlsx can take 1-2 minutes
# because openpyxl must parse the XML structure.
# Skip this cell if you're happy with the CSV exploration above.

xlsx_sample = pd.read_excel(XLSX_PATH, nrows=5, engine="openpyxl")
print(f"Columns ({len(xlsx_sample.columns)}):")
print(list(xlsx_sample.columns))
xlsx_sample

c:\Users\roman\Envs\globalenv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Columns (59):
['OBJECTID', 'PCD7', 'PCD8', 'PCDS', 'DOINTR', 'DOTERM', 'CTY25CD', 'CED25CD', 'LAD25CD', 'WD25CD', 'PARNCP25CD', 'USRTYPIND', 'EAST1M', 'NORTH1M', 'GRIDIND', 'HLTH19CD', 'NHSER24CD', 'CTRY25CD', 'RGN25CD', 'SSR95CD', 'PCON24CD', 'EER20CD', 'EDUC23CD', 'TTWA15CD', 'PCO19CD', 'ITL25CD', 'WDSTL05CD', 'OA01CD', 'WDCAS03CD', 'NPARK16CD', 'LSOA01CD', 'MSOA01CD', 'RUC01IND', 'OAC01IND', 'OA11CD', 'LSOA11CD', 'MSOA11CD', 'WZ11CD', 'SICBL24CD', 'BUA24CD', 'RUC11IND', 'OAC11IND', 'LAT', 'LONG', 'LEP21CD1', 'LEP21CD2', 'PFA23CD', 'IMD20IND', 'CAL24CD', 'ICB23CD', 'OA21CD', 'LSOA21CD', 'MSOA21CD', 'RUC21IND', '_matched_characters', '_matched_parts___part', '_matched_parts___startIndex', 'x', 'y']


,OBJECTID,PCD7,PCD8,PCDS,DOINTR,DOTERM,CTY25CD,CED25CD,LAD25CD,WD25CD,...,ICB23CD,OA21CD,LSOA21CD,MSOA21CD,RUC21IND,_matched_characters,_matched_parts___part,_matched_parts___startIndex,x,y
0,1,AB101AB,AB10 1AB,AB10 1AB,201106,NaN,S99999999,S99999999,S12000033,S13002842,...,S99999999,S00136377,S01013627,S02002541,1,NaN,NaN,NaN,-2.096899,57.149674
1,2,AB101AF,AB10 1AF,AB10 1AF,199606,NaN,S99999999,S99999999,S12000033,S13002842,...,S99999999,S00136377,S01013627,S02002541,1,NaN,NaN,NaN,-2.096899,57.149674
2,3,AB101AG,AB10 1AG,AB10 1AG,199606,NaN,S99999999,S99999999,S12000033,S13002842,...,S99999999,S00136377,S01013627,S02002541,1,NaN,NaN,NaN,-2.096980,57.149135
3,4,AB101AH,AB10 1AH,AB10 1AH,199606,NaN,S99999999,S99999999,S12000033,S13002842,...,S99999999,S00136377,S01013627,S02002541,1,NaN,NaN,NaN,-2.096899,57.149674
4,5,AB101AL,AB10 1AL,AB10 1AL,199606,NaN,S99999999,S99999999,S12000033,S13002842,...,S99999999,S00136377,S01013627,S02002541,1,NaN,NaN,NaN,-2.095892,57.150142


## 3. Load Lookup Tables

These are small files (< 12K rows each) that map area codes → human-readable names. We load them fully into memory and use them to join place names onto postcodes.

In [6]:
# Ward names (8,407 rows) — neighbourhood-level names like "Farnham Castle", "Clifton"
wd = pd.read_csv(DOCS / "WD Ward names and codes UK as at 05_25.csv", encoding="utf-8-sig")
print(f"Wards: {len(wd)} rows")
wd.head()

Wards: 8407 rows


,WD25CD,WD25NM,WD25NMW
0,E05000932,Ainsdale,NaN
1,E05000933,Birkdale,NaN
2,E05000934,Blundellsands,NaN
3,E05000935,Cambridge,NaN
4,E05000936,Church,NaN


In [7]:
# Local Authority District names (361 rows) — council-level names like "Waverley", "Bristol"
lad = pd.read_csv(DOCS / "LAD Local Authority District names and codes UK as at 04_25.csv", encoding="utf-8-sig")
print(f"LADs: {len(lad)} rows")
lad.head()

LADs: 361 rows


,LAD25CD,LAD25NM,LAD25NMW
0,E06000001,Hartlepool,NaN
1,E06000002,Middlesbrough,NaN
2,E06000003,Redcar and Cleveland,NaN
3,E06000004,Stockton-on-Tees,NaN
4,E06000005,Darlington,NaN


In [8]:
# Built-Up Area names (7,776 rows) — town/city names like "London", "Farnham", "Bristol"
bua = pd.read_csv(DOCS / "BUA Built Up Area names and codes EW as at 04_24.csv", encoding="utf-8-sig")
print(f"BUAs: {len(bua)} rows")
bua.head()

BUAs: 7776 rows


,BUA24CD,BUA24NM,BUA24NMW
0,E63007092,Berwick-upon-Tweed,NaN
1,E63007093,Loanend,NaN
2,E63007094,Norham,NaN
3,E63007095,Haggerston,NaN
4,E63007096,Lowick,NaN


In [9]:
# Parish names (11,539 rows) — England & Wales only, e.g. "Farnham", "Claxton"
parish = pd.read_csv(DOCS / "PARNCP Parish_Non_Civil Parish names and codes ew as at 04_25 v2.csv", encoding="utf-8-sig")
print(f"Parishes: {len(parish)} rows")
parish.head()

Parishes: 11539 rows


,PARNCP25CD,PARNCP25NM,PARNCP25NMW,LAD25CD,LAD25NM,LAD25NMW
0,E04000253,Brierton,NaN,E06000001,Hartlepool,NaN
1,E04000254,Claxton,NaN,E06000001,Hartlepool,NaN
2,E04000255,Dalton Piercy,NaN,E06000001,Hartlepool,NaN
3,E04000258,Hart,NaN,E06000001,Hartlepool,NaN
4,E04000259,Newton Bewley,NaN,E06000001,Hartlepool,NaN


In [10]:
# Build quick lookup dicts (code → name) for joining
wd_lookup = wd.set_index("WD25CD")["WD25NM"].to_dict()
lad_lookup = lad.set_index("LAD25CD")["LAD25NM"].to_dict()
bua_lookup = bua.set_index("BUA24CD")["BUA24NM"].to_dict()
parish_lookup = parish.set_index("PARNCP25CD")["PARNCP25NM"].to_dict()

print(f"Lookup sizes — Wards: {len(wd_lookup)}, LADs: {len(lad_lookup)}, BUAs: {len(bua_lookup)}, Parishes: {len(parish_lookup)}")

Lookup sizes — Wards: 8407, LADs: 361, BUAs: 7775, Parishes: 11539


## 4. Process the Main CSV in Batches

Read the 2.7M row CSV in chunks of 100K rows. For each chunk:
1. Filter to **live postcodes only** (`doterm` is empty = not terminated)
2. Drop rows with missing lat/lng (e.g. PO boxes, non-geographic postcodes)
3. Map area codes to human-readable names via lookup dicts
4. Keep only the columns we need

The result is a clean DataFrame with: `postcode`, `lat`, `lng`, `ward`, `district`, `built_up_area`, `parish`

In [11]:
CHUNK_SIZE = 100_000
USE_COLS = ["pcds", "lat", "long", "doterm", "lad25cd", "wd25cd", "bua24cd", "parncp25cd"]

chunks = []
total_read = 0
total_kept = 0

for chunk in pd.read_csv(CSV_PATH, usecols=USE_COLS, chunksize=CHUNK_SIZE, dtype=str):
    total_read += len(chunk)
    
    # Keep only live postcodes (doterm is empty/NaN = still active)
    live = chunk[chunk["doterm"].isna() | (chunk["doterm"] == "")]
    
    # Convert lat/long to float, drop rows without coordinates
    live = live.copy()
    live["lat"] = pd.to_numeric(live["lat"], errors="coerce")
    live["long"] = pd.to_numeric(live["long"], errors="coerce")
    live = live.dropna(subset=["lat", "long"])
    
    # Drop postcodes at 0,0 (invalid/placeholder)
    live = live[(live["lat"] != 0) & (live["long"] != 0)]
    
    # Map codes to names
    live["ward"] = live["wd25cd"].map(wd_lookup)
    live["district"] = live["lad25cd"].map(lad_lookup)
    live["built_up_area"] = live["bua24cd"].map(bua_lookup)
    live["parish"] = live["parncp25cd"].map(parish_lookup)
    
    # Rename and keep only what we need
    live = live.rename(columns={"pcds": "postcode", "long": "lng"})
    live = live[["postcode", "lat", "lng", "ward", "district", "built_up_area", "parish"]]
    
    chunks.append(live)
    total_kept += len(live)
    
    print(f"  Processed {total_read:,} rows → kept {total_kept:,} live postcodes with coords", end="\r")

print(f"\nDone! Read {total_read:,} total rows, kept {total_kept:,} live postcodes with coordinates.")

  Processed 2,717,743 rows → kept 1,793,065 live postcodes with coords
Done! Read 2,717,743 total rows, kept 1,793,065 live postcodes with coordinates.


In [12]:
# Combine all chunks into one DataFrame
df = pd.concat(chunks, ignore_index=True)
del chunks  # free memory

print(f"Final shape: {df.shape}")
print(f"\nNull counts per column:")
print(df.isnull().sum())
print(f"\nSample rows:")
df.sample(10, random_state=42)

Final shape: (1793065, 7)

Null counts per column:
postcode              0
lat                   0
lng                   0
ward                  0
district              0
built_up_area    402576
parish           210623
dtype: int64

Sample rows:


,postcode,lat,lng,ward,district,built_up_area,parish
985212,MK43 7DG,52.200723,-0.607148,Harrold,Bedford,Harrold,Harrold
70659,BA12 7QB,51.225009,-2.252926,Warminster North & Rural,Wiltshire,NaN,Chapmanslade
880585,LL49 9PD,52.927020,-4.134410,Gorllewin Porthmadog,Gwynedd,Porthmadog,Porthmadog
1780712,YO22 5DB,54.453126,-0.659717,Esk Valley & Coast,North Yorkshire,Sleights,Eskdaleside cum Ugglebarnby
1518054,ST10 4PH,52.915832,-2.006247,Blythe,East Staffordshire,NaN,Leigh
43449,B44 0TA,52.549633,-1.870763,Kingstanding,Birmingham,Birmingham,"Birmingham, unparished area"
497403,DY13 0NT,52.323386,-2.295315,Areley Kings & Riverside,Wyre Forest,Stourport-on-Severn,Stourport-on-Severn
510502,E14 8BS,51.510247,-0.031977,Limehouse,Tower Hamlets,Tower Hamlets,"Tower Hamlets, unparished area"
34319,B26 3ST,52.457553,-1.772387,Sheldon,Birmingham,Birmingham,"Birmingham, unparished area"
1003674,N12 0AE,51.603208,-0.173479,Woodhouse,Barnet,Barnet,"Barnet, unparished area"


## 5. Derive a "best place name" per postcode

The ONS data gives us multiple name layers: ward, district, built-up area, parish. We pick the **best available** name for each postcode using this priority:

1. **Built-up area** (best — actual town/city names like "Farnham", "London")
2. **Parish** (good — village/town level, England & Wales only)
3. **Ward** (fallback — neighbourhood names, always available)
4. **District** (last resort — council names like "Waverley", "Bristol")

In [13]:
# Pick best available place name per row
df["place_name"] = (
    df["built_up_area"]
    .fillna(df["parish"])
    .fillna(df["ward"])
    .fillna(df["district"])
)

print(f"Place name coverage:")
print(f"  From built_up_area: {df['built_up_area'].notna().sum():,}")
print(f"  From parish:        {(df['place_name'] == df['parish']).sum() - df['built_up_area'].notna().sum():,}")
print(f"  From ward:          {df[df['built_up_area'].isna() & df['parish'].isna() & df['ward'].notna()].shape[0]:,}")
print(f"  From district:      {df[df['built_up_area'].isna() & df['parish'].isna() & df['ward'].isna() & df['district'].notna()].shape[0]:,}")
print(f"  Still missing:      {df['place_name'].isna().sum():,}")
print()

# Show examples where place_name came from each source
print("Examples:")
df[["postcode", "place_name", "built_up_area", "parish", "ward", "district"]].sample(10, random_state=7)

Place name coverage:
  From built_up_area: 1,390,489
  From parish:        -828,930
  From ward:          210,623
  From district:      0
  Still missing:      0

Examples:


,postcode,place_name,built_up_area,parish,ward,district
633367,GL14 9GU,Cinderford,Cinderford,Cinderford,Cinderford East,Forest of Dean
232901,BT66 7NL,Donaghcloney,NaN,NaN,Donaghcloney,"Armagh City, Banbridge and Craigavon"
454855,DL13 5SB,Lynesack and Softley,NaN,Lynesack and Softley,Evenwood,County Durham
112057,BD7 3LE,Bradford,Bradford,"Bradford, unparished area",Great Horton,Bradford
288583,CF64 1EH,Penarth,Penarth,Penarth,St Augustine's,Vale of Glamorgan
862613,LE67 5HZ,Whitwick,Whitwick,Whitwick,Hermitage,North West Leicestershire
1776577,YO14 9DJ,Filey,Filey,Filey,Filey,North Yorkshire
108469,BD24 0DF,Giggleswick,Giggleswick,Giggleswick,Settle & Penyghent,North Yorkshire
1413239,SG10 6BS,Hadham Cross,Hadham Cross,Much Hadham,Much Hadham,East Hertfordshire
451718,DL1 9GD,Darlington,Darlington,"Darlington, unparished area",Park East,Darlington


## 6. Export: Postcodes CSV (for PocketBase `postcodes` collection)

Minimal file: `postcode, lat, lng` — this is what gets imported into PocketBase for instant local lookups.

In [14]:
out_postcodes = df[["postcode", "lat", "lng"]].copy()

OUT_DIR = BASE / "data_exports"
OUT_DIR.mkdir(exist_ok=True)

out_postcodes.to_csv(OUT_DIR / "postcodes.csv", index=False)
print(f"Saved {len(out_postcodes):,} postcodes to {OUT_DIR / 'postcodes.csv'}")
print(f"File size: {(OUT_DIR / 'postcodes.csv').stat().st_size / 1e6:.1f} MB")
out_postcodes.head()

Saved 1,793,065 postcodes to c:\Users\roman\Documents\projects\UK_Music_Experiences\data_exports\postcodes.csv
File size: 52.2 MB


,postcode,lat,lng
0,AB10 1AB,57.149590,-2.096923
1,AB10 1AF,57.149590,-2.096923
2,AB10 1AG,57.149051,-2.097004
3,AB10 1AH,57.149590,-2.096923
4,AB10 1AL,57.149591,-2.095304


## 7. Export: Places CSV (for PocketBase `places` collection)

Derive unique place names with their average lat/lng (centroid of all postcodes in that place). This gives us a local lookup for when users type "Farnham" or "London" instead of a postcode.

In [15]:
# Build places from multiple name layers, each contributing unique place names with centroids

places_parts = []

# Built-up areas (best quality — actual town/city names)
bua_places = (
    df[df["built_up_area"].notna()]
    .groupby("built_up_area")
    .agg(lat=("lat", "mean"), lng=("lng", "mean"), count=("postcode", "size"))
    .reset_index()
    .rename(columns={"built_up_area": "name"})
)
bua_places["source"] = "built_up_area"
places_parts.append(bua_places)

# Parishes (village/town level)
parish_places = (
    df[df["parish"].notna()]
    .groupby("parish")
    .agg(lat=("lat", "mean"), lng=("lng", "mean"), count=("postcode", "size"))
    .reset_index()
    .rename(columns={"parish": "name"})
)
parish_places["source"] = "parish"
places_parts.append(parish_places)

# Wards (neighbourhood level)
ward_places = (
    df[df["ward"].notna()]
    .groupby("ward")
    .agg(lat=("lat", "mean"), lng=("lng", "mean"), count=("postcode", "size"))
    .reset_index()
    .rename(columns={"ward": "name"})
)
ward_places["source"] = "ward"
places_parts.append(ward_places)

# Districts (council level)
district_places = (
    df[df["district"].notna()]
    .groupby("district")
    .agg(lat=("lat", "mean"), lng=("lng", "mean"), count=("postcode", "size"))
    .reset_index()
    .rename(columns={"district": "name"})
)
district_places["source"] = "district"
places_parts.append(district_places)

all_places = pd.concat(places_parts, ignore_index=True)

# Deduplicate: if the same name appears at multiple levels, keep the one with the most postcodes
# (e.g. "Farnham" as BUA is better than "Farnham" as ward)
# Sort by source priority, then drop duplicates keeping first
source_priority = {"built_up_area": 0, "parish": 1, "ward": 2, "district": 3}
all_places["priority"] = all_places["source"].map(source_priority)
all_places = all_places.sort_values(["name", "priority"]).drop_duplicates(subset="name", keep="first")
all_places = all_places.drop(columns="priority")

# Round coordinates
all_places["lat"] = all_places["lat"].round(6)
all_places["lng"] = all_places["lng"].round(6)

print(f"Unique places: {len(all_places):,}")
print(f"\nBy source:")
print(all_places["source"].value_counts())
print(f"\nSample:")
all_places.sample(15, random_state=42)

Unique places: 20,865

By source:
source
built_up_area    7748
parish           6658
ward             6259
district          200
Name: count, dtype: int64

Sample:


,name,lat,lng,count,source
18577,Amroth and Saundersfoot North,51.733131,-4.681183,114,ward
25918,West Marsh,53.569006,-0.090534,353,ward
2928,Gun Green,51.049789,0.534682,3,built_up_area
13444,Llandissilio West,51.860250,-4.731717,29,parish
16093,Sourton,50.700741,-4.080345,27,parish
24640,Shettleston,55.844332,-4.165545,780,ward
10244,Cransley,52.386530,-0.772863,13,parish
22735,Livingston South,55.881334,-3.517569,455,ward
3000,Ham,51.365393,-1.526488,9,built_up_area
15003,Penylan,51.499822,-3.158255,295,parish


In [16]:
# Sanity check: can we find key places?
test_names = ["Farnham", "London", "Bristol", "Manchester", "Edinburgh", "Brighton"]
for name in test_names:
    match = all_places[all_places["name"].str.lower() == name.lower()]
    if len(match):
        row = match.iloc[0]
        print(f"  {name}: lat={row['lat']:.4f}, lng={row['lng']:.4f} (source: {row['source']}, {row['count']} postcodes)")
    else:
        print(f"  {name}: NOT FOUND")

  Farnham: lat=51.2095, lng=-0.7957 (source: built_up_area, 700 postcodes)
  London: NOT FOUND
  Bristol: lat=51.4578, lng=-2.5932 (source: built_up_area, 10522 postcodes)
  Manchester: lat=53.4714, lng=-2.2204 (source: built_up_area, 12079 postcodes)
  Edinburgh: NOT FOUND
  Brighton: NOT FOUND


In [17]:
# Save places CSV
out_places = all_places[["name", "source", "lat", "lng", "count"]].sort_values("name")
out_places.to_csv(OUT_DIR / "places.csv", index=False)
print(f"Saved {len(out_places):,} places to {OUT_DIR / 'places.csv'}")
print(f"File size: {(OUT_DIR / 'places.csv').stat().st_size / 1e6:.1f} MB")

Saved 20,865 places to c:\Users\roman\Documents\projects\UK_Music_Experiences\data_exports\places.csv
File size: 1.0 MB


## 8. Also export a combined "full" CSV

One file with all columns for reference/debugging: `postcode, lat, lng, place_name, ward, district, built_up_area, parish`

In [18]:
out_full = df[["postcode", "lat", "lng", "place_name", "ward", "district", "built_up_area", "parish"]].copy()
out_full.to_csv(OUT_DIR / "postcodes_full.csv", index=False)
print(f"Saved {len(out_full):,} rows to {OUT_DIR / 'postcodes_full.csv'}")
print(f"File size: {(OUT_DIR / 'postcodes_full.csv').stat().st_size / 1e6:.1f} MB")
print(f"\nAll output files:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

Saved 1,793,065 rows to c:\Users\roman\Documents\projects\UK_Music_Experiences\data_exports\postcodes_full.csv
File size: 175.0 MB

All output files:
  places.csv  (1.0 MB)
  postcodes.csv  (52.2 MB)
  postcodes_full.csv  (175.0 MB)


## Summary

**Output files** (in `data_exports/`):

| File | Purpose | Rows | For PocketBase collection |
|------|---------|------|--------------------------|
| `postcodes.csv` | postcode, lat, lng | ~1.7M | `postcodes` |
| `places.csv` | name, source, lat, lng, count | ~15K–25K | `places` |
| `postcodes_full.csv` | All columns including all name layers | ~1.7M | Reference only |

**Next step:** Import `postcodes.csv` and `places.csv` into PocketBase via Admin API bulk-insert script.